In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="MohamedRashad/Quran-Recitations", 
                  repo_type="dataset", local_dir="./Quran-Recitations")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 102 files: 100%|██████████| 102/102 [00:00<00:00, 2966.47it/s]


'/home/ubuntu/Quran-Recitations'

In [3]:
files = glob('Quran-Recitations/*/*.parquet')
len(files)

100

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['text'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': t,
                    'speaker': f"{base}_{df['source'].iloc[i]}"
                })
            except Exception as e:
                pass
        
    return data

In [5]:
# data = loop((files[:1], 0))
# data

In [8]:
# data = multiprocessing(files, loop, cores = 30)

In [9]:
len(data)

113283

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Quran-Recitations_audio/Quran-Recitations-data-train-00090-of-00100_0.mp3',
 'text': 'إِنَّ ٱلَّذِينَ كَفَرُوا۟ سَوَآءٌ عَلَيْهِمْ ءَأَنذَرْتَهُمْ أَمْ لَمْ تُنذِرْهُمْ لَا يُؤْمِنُونَ',
 'speaker': 'Quran-Recitations_audio_محمد صديق المنشاوي (المجود)'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Quran-Recitations')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 14.71ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 11.5MB / 11.5MB, 1.20MB/s  
Processing Files (1 / 1): 100%|██████████| 11.5MB / 11.5MB, 1.18MB/s  
Processing Files (1 / 1): 100%|██████████| 11.5MB / 11.5MB, 1.15MB/s  
New Data Upload: 100%|██████████| 11.4MB / 11.4MB, 1.14MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.53s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/625994e2fcf81305c6244a1bf29e8c51e6fddee0', commit_message='Upload dataset', commit_description='', oid='625994e2fcf81305c6244a1bf29e8c51e6fddee0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('Quran-Recitations-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
!du -hs Quran-Recitations_audio

17G	Quran-Recitations_audio


In [18]:
!zip -rq Quran-Recitations_audio.zip Quran-Recitations_audio

In [16]:
!zip -rq Quran-Recitations_audio_neucodec.zip Quran-Recitations_audio_neucodec

In [21]:
# from huggingface_hub import HfApi
# api = HfApi()

# for f in glob('Quran-Recitations_audio*.zip'):
#     print(f)
#     api.upload_file(
#         path_or_fileobj=f,
#         path_in_repo=f,
#         repo_id="malaysia-ai/Multilingual-TTS",
#         repo_type="dataset",
#     )